# Creating Your First Agent

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) LangChain roadmap.

You will learn how to create an agent that can reason about which tools to call and when.

## 1. Install Dependencies

In [ ]:
!pip install -q langchain "langchain[openai]" langgraph

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Define a Tool

Create a simple weather tool that the agent can call.

In [ ]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    return f"The weather in {city} is 72°F and sunny."

## 4. Create the Agent

Use `create_react_agent` to combine a model, tools, and a system prompt.

In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent

model = init_chat_model("gpt-4o-mini", model_provider="openai")

agent = create_react_agent(
    model=model,
    tools=[get_weather],
    prompt="You are a helpful weather assistant. Use the get_weather tool to answer weather questions.",
)

## 5. Invoke the Agent

Send a message and observe how the agent reasons and calls the tool.

In [ ]:
from langchain_core.messages import HumanMessage

result = agent.invoke({"messages": [HumanMessage(content="What's the weather in Tokyo?")]})

for msg in result["messages"]:
    print(f"[{msg.type}] {msg.content}")

## 6. Understanding the Agent Loop

The agent follows this cycle:

1. **LLM decides** — respond directly or call a tool
2. **Tool executes** — if a tool was called, it runs and returns a `ToolMessage`
3. **LLM sees result** — decides again: respond or call another tool
4. **Loop ends** — when the LLM produces a final text response

Let's see the full message trace.

In [ ]:
result = agent.invoke({"messages": [HumanMessage(content="What's the weather in Paris?")]})

for i, msg in enumerate(result["messages"]):
    print(f"Message {i} [{msg.type}]:")
    if msg.content:
        print(f"  Content: {msg.content}")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  Tool call: {tc['name']}({tc['args']})")
    print()

## 7. Non-Tool Questions

When the user's question doesn't require a tool, the agent responds directly.

In [ ]:
result = agent.invoke({"messages": [HumanMessage(content="What is a haiku?")]})
print(result["messages"][-1].content)

## Summary

- `create_react_agent` builds an agent from a model, tools, and a prompt
- The agent loop cycles: LLM reason → tool call → observe → repeat
- Results contain the full message history including tool interactions
- The agent decides whether to call tools or respond directly